# Carrier-free constant-|B| growth (v2, `grow.py`)

Grows a large-amplitude, genuinely 3D, constant-$|B|$, divergence-free field
from a uniform mean field plus an arbitrary divergence-free seed, using the
$\mu$-only exact-Galerkin Gauss--Newton solver (`constantB/solver_mu.py`;
see `docs/algorithm.tex` for the method and `docs/SPEC_v2.md` for the code spec).

**Accelerator: GPU P100** (float64 required; T4 fp64 is ~1/32 fp32).


In [ ]:
# MUST run before any process touches CUDA: JAX preallocates ~75% of device
# memory by default; grow.py runs as a subprocess and would collide with the
# verification cell below.
%env XLA_PYTHON_CLIENT_PREALLOCATE=false
%env XLA_PYTHON_CLIENT_ALLOCATOR=platform


In [ ]:
import subprocess
name = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).decode().strip()
print("Detected GPU:", name)
if "P100" not in name:
    print("\n*** WARNING: not a P100 (%s). float64 throughput will be poor. ***" % name)


In [ ]:
!pip install -q -U "jax[cuda12]"


In [ ]:
import jax
jax.config.update("jax_enable_x64", True)  # constantB does this too; standalone check
import jax.numpy as jnp
print("jax", jax.__version__, jax.devices())
print("x64 check (expect float64):", jnp.array(1.0).dtype)
assert jax.devices()[0].platform == "gpu"


In [ ]:
import os
REPO_DIR = "/kaggle/working/jax_constantB"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/alfredmallet/jax_constantB.git {REPO_DIR}
else:
    print("repo present, skipping clone")


## The run

Blob seed on a uniform $\hat z$ mean field, Galerkin ladder up to $96^2\times192$.
Resumable: rerunning the cell continues from `/kaggle/working/grow.npz`.
Switch `--seed random --kmax 4 --key 1` for an arbitrary random-phase seed
(`kmax` must sit strictly inside the grid0 band, $k_{max} < \min(N)/3$ — enforced);
`--freeze-top 3` pins the seed's three dominant (non-coplanar) modes, activating
at `--freeze-after` (default $\varepsilon=0.2$; pinning from the uniform start is infeasible).

**Notes.** (i) This clones from GitHub: the v2 stack must be pushed there first.
(ii) Blob seeds are analytic but not band-limited, so ladder rungs rebuild the blob with
$\sim10^{-3}$ drift at coarse grids ($10^{-14}$ by $48^2\times96$); for strict
same-continuum-object ladders use `--seed random` or `--seed file`.


In [ ]:
%cd /kaggle/working/jax_constantB
!XLA_PYTHON_CLIENT_PREALLOCATE=false XLA_PYTHON_CLIENT_ALLOCATOR=platform \
  python3 -u grow.py \
    --state /kaggle/working/grow.npz --csv /kaggle/working/grow.csv \
    --grid0 24 24 48 --grid-max 96 96 192 \
    --seed blob --w 0.8 0.8 0.8 --kz 1 \
    --eps-max 3.0 --de 0.03 --sweeps 8 --cgit 800 \
    --res-ok 1e-9 --gtail-max 1e-3 --smooth 0.0 \
    --snap-de 0.25 --max-seconds 39600 \
    2>&1 | tee -a /kaggle/working/grow_log.txt


## Figures + verification


In [ ]:
%cd /kaggle/working/jax_constantB
!python3 grow.py --state /kaggle/working/grow.npz --csv /kaggle/working/grow.csv --plot
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("/kaggle/working/*.png")) + sorted(glob.glob("fig_grow*.png")):
    display(Image(f))


In [ ]:
import sys; sys.path.insert(0, "/kaggle/working/jax_constantB")
import numpy as np
from constantB.state_io import load_state
from constantB.solver_mu import MuSolver
from constantB.spectra import kspace_inertia
B, eps, meta = load_state("/kaggle/working/grow.npz")
S = MuSolver(B.shape[1:], smooth=float(meta.get("smooth", 0.0)))
d, q = S.residual(B)
print(f"eps={eps:.3f} grid={B.shape[1:]}")
print(f"max|div B| {float(abs(np.asarray(d)).max()):.2e}  max|Tq| {float(abs(np.asarray(q)).max()):.2e}")
print("tail_norm (rms, max):", S.tail_norm(B))
print("k-space inertia (3D-ness):", kspace_inertia(B))
